In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import hdfstream 
import astropy.units as u
import scipy
import lightcone_io as lc
import unyt
from astropy import constants as const

In [ ]:
#Sets up the connection to the data and reads in the shell array
root_dir = hdfstream.open("cosma", "/")
basedir="FLAMINGO/L1_m9/L1_m9/healpix_maps/nside_4096"
basename="lightcone0"
shell = lc.ShellArray(basedir, basename, remote_dir=root_dir)

In [ ]:
# Index of the shell we're interested in
shell_nr = 0

# Name of the quantity we want to read
map_name = "TotalMass"

# Read the data
map_data = shell[shell_nr][map_name][...]

In [ ]:
def my_mean(data):
    return np.mean(data)

In [ ]:
# 1. Define the physical constants (using astropy for accuracy)
h = const.h.value       # Planck constant in J*s
k_B = const.k_B.value   # Boltzmann constant in J/K
T_cmb = 2.7255
freq_test = 150.0 # in GHz

x = h*freq_test*1e9/(T_cmb*k_B)
tSZ_freq_dep_func = x*(np.cosh(x/2)/np.sinh(x/2))-4

with hdfstream.open("cosma", "FLAMINGO/L1_m9/L1_m9/integrated_maps/yang26/lightcone0_shells/lensed_tSZ_rot_same_rot.hdf5") as map_file:
  lensed_tSZ_map = map_file["data"][...]
delta_T_tSZ = tSZ_freq_dep_func * lensed_tSZ_map * T_cmb ## in K_CMB

In [ ]:
file_path = 'FLAMINGO/L1_m9/L1_m9/integrated_maps/broxterman24/WL_convergence_Euclid_like_nz_Broxterman24_L1_m9_lc0.hdf5'
with hdfstream.open("cosma", file_path) as data_file:
  kappa_map = data_file["Convergence"][:]

In [ ]:
filename = "FLAMINGO/L1_m9/L1_m9/halo_lightcone/lightcone0/lightcone_halos_0075.hdf5"
halos = lc.HaloLightconeFile(filename, remote_dir=root_dir)
# List of halo properties to read
properties = ("Lightcone/HaloCentre", "BoundSubhalo/TotalMass", "Lightcone/Redshift")

# Read the data
halo_props = halos.read_halos(properties)

In [ ]:
mask = (halo_props["BoundSubhalo/TotalMass"] > 1e5)

In [ ]:
# --- 2. Isolate the clusters for the 3 bins ---
# The first 3 bins each contain 100 clusters, using the largest clusters.
items_per_bin = 100
num_bins = 3

# Convert mass data to plain numpy floats in solar masses
masses = halo_props["BoundSubhalo/TotalMass"].to('Msun').value
sorted_masses = np.sort(masses)

# Use the largest 300 clusters for the three bins
largest_clusters = sorted_masses[-items_per_bin * num_bins:]

# Split the largest 300 clusters into 3 equal-count bins
quantiles = np.linspace(0, 100, num_bins + 1)
bin_edges = np.percentile(largest_clusters, quantiles)

# Keep the 3-bin structure simple by plotting only these 300 clusters
# with exactly 4 edges (which creates 3 bins)
top_clusters = largest_clusters

# --- 3. Plot the Histogram ---
plt.figure(figsize=(10, 6))
counts, edges, bars = plt.hist(top_clusters, bins=bin_edges, edgecolor='black', color='crimson', alpha=0.8)

# Add exact count labels on top of each bin bar
plt.bar_label(bars, fmt='%d', padding=3, weight='bold')

# Log scale for the x-axis is necessary to visualize wide mass gaps clearly
plt.xscale('log')

plt.xlabel(r'Cluster Mass ($M_{\odot}$)', fontsize=12)
plt.ylabel('Number of Galaxy Clusters', fontsize=12)
plt.grid(axis='both', which='both', linestyle='--', alpha=0.4)

# Set y-limit with padding so labels don't get cut off
plt.ylim(0, max(counts) + 5)

# --- 4. Print out the verification ---
print("Final Bin Breakdown:")
for i in range(len(bin_edges) - 1):
    print(f"Bin {i+1} ({edges[i]:.2e} to {edges[i+1]:.2e}): {int(counts[i])} clusters")

In [ ]:
bin_edges_0sigma = bin_edges

In [ ]:
fov_deg = 1
pixel_scale_arcmin = 0.01
npix = int(fov_deg * 60 / pixel_scale_arcmin)

In [ ]:
halo_x = halo_props["Lightcone/HaloCentre"][mask][:, 0]
halo_y = halo_props["Lightcone/HaloCentre"][mask][:, 1]
halo_z = halo_props["Lightcone/HaloCentre"][mask][:, 2] 

ra_deg = np.arctan2(halo_y, halo_x) * 180 / np.pi
dec_deg = np.arcsin(halo_z / np.sqrt(halo_x**2 + halo_y**2 + halo_z**2)) * 180 / np.pi

In [ ]:
tsz_map = lensed_tSZ_map - np.mean(lensed_tSZ_map)

In [ ]:
tsz_list = []
kappa_list = []
for i in range(len(ra_deg)):
    tsz_patch = hp.gnomview(tsz_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = "tSZ", return_projected_map=True, cmap = 'plasma', no_plot=True)
    kappa_patch = hp.gnomview(kappa_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = r"$\kappa$",return_projected_map=True, no_plot=True)
    tsz_list.append(tsz_patch)
    kappa_list.append(kappa_patch)


In [ ]:
def radial_profile(data, center = None, nbins = 20):
    if center is None:
        center = (data.shape[1]//2, data.shape[0]//2)
    y, x = np.indices((data.shape))
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2)
    #r = r.astype(np.int)
    r_flat = r.flatten()
    image_flat = data.flatten()
    
    rmax = np.max(r)
    bins = np.linspace(0, rmax, nbins+1)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    profile_sum, _ = np.histogram(r_flat, bins=bins, weights=image_flat)
    pixel_count, _ = np.histogram(r_flat, bins=bins)
    radial_profile = np.divide(profile_sum, pixel_count, out=np.zeros_like(profile_sum), where=pixel_count!=0)
    
    return bin_centers, radial_profile

In [ ]:
tsz_scatter_0_sigma = scipy.stats.bootstrap(my_mean((tsz_list)), np.mean,n_resamples = 1000, axis = 0)
kappa_scatter_0_sigma = scipy.stats.bootstrap(my_mean((kappa_list)), np.mean, n_resamples = 1000, axis = 0)

In [ ]:
tsz_stack_0_sigma = np.mean(tsz_list, axis = 0)
kappa_stack_0_sigma = np.mean(kappa_list, axis = 0)

In [ ]:
r_tsz_0_sigma, prof_tsz_0_sigma = radial_profile(tsz_stack_0_sigma)
r_kappa_0_sigma, prof_kappa_0_sigma = radial_profile(kappa_stack_0_sigma)

In [ ]:
plt.plot(r_tsz_0_sigma, prof_tsz_0_sigma/np.max(prof_tsz_0_sigma), label = "tSZ")
plt.plot(r_kappa_0_sigma, prof_kappa_0_sigma/np.max(prof_kappa_0_sigma), label = r"$\kappa$") 
#plt.yscale('log') 
plt.legend()

In [ ]:
#Opening the data for the 8sigma case
basedir="FLAMINGO/L1_m9/fgas-8sigma/healpix_maps/nside_4096"
basename="lightcone0"
shell = lc.ShellArray(basedir, basename, remote_dir=root_dir)

file_path = 'FLAMINGO/L1_m9/fgas-8sigma/integrated_maps/broxterman24/WL_convergence_Euclid_like_nz_Broxterman24_fgas-8sigma_lc0.hdf5'
with hdfstream.open("cosma", file_path) as data_file:
  kappa_map = data_file["Convergence"][:]

filename = "FLAMINGO/L1_m9/fgas-8sigma/halo_lightcone/lightcone0/lightcone_halos_0075.hdf5"
halos = lc.HaloLightconeFile(filename, remote_dir=root_dir)

x = h*freq_test*1e9/(T_cmb*k_B)
tSZ_freq_dep_func = x*(np.cosh(x/2)/np.sinh(x/2))-4

with hdfstream.open("cosma", "FLAMINGO/L1_m9/fgas-8sigma/integrated_maps/yang26/lightcone0_shells/lensed_tSZ_rot_same_rot.hdf5") as map_file:
  lensed_tSZ_map = map_file["data"][...]
delta_T_tSZ = tSZ_freq_dep_func * lensed_tSZ_map * T_cmb ## in K_CMB

# List of halo properties to read
properties = ("Lightcone/HaloCentre", "BoundSubhalo/TotalMass", "Lightcone/Redshift")

# Read the data
halo_props = halos.read_halos(properties)

In [ ]:
mask = (halo_props["BoundSubhalo/TotalMass"] > 1e5)
halo_x = halo_props["Lightcone/HaloCentre"][mask][:, 0]
halo_y = halo_props["Lightcone/HaloCentre"][mask][:, 1]
halo_z = halo_props["Lightcone/HaloCentre"][mask][:, 2] 
ra_deg = np.arctan2(halo_y, halo_x) * 180 / np.pi
dec_deg = np.arcsin(halo_z / np.sqrt(halo_x**2 + halo_y**2 + halo_z**2)) * 180 / np.pi

In [ ]:
tsz_map = lensed_tSZ_map - np.mean(lensed_tSZ_map)
tsz_list = []
kappa_list = []
for i in range(len(ra_deg)):
    tsz_patch = hp.gnomview(tsz_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = "tSZ", return_projected_map=True, cmap = 'plasma', no_plot=True)
    kappa_patch = hp.gnomview(kappa_map, rot=(ra_deg[i], dec_deg[i]), xsize=npix, reso = pixel_scale_arcmin, title = r"$\kappa$",return_projected_map=True, no_plot=True)
    tsz_list.append(tsz_patch)
    kappa_list.append(kappa_patch)

In [ ]:
tsz_scatter_8_sigma = bootstrap(tsz_list)
kappa_scatter_8_sigma = bootstrap(kappa_list)
tsz_stack_8_sigma = np.mean(tsz_list, axis = 0)
kappa_stack_8_sigma = np.mean(kappa_list, axis = 0)
_ , prof_scatter_tsz_8_sigma = radial_profile(tsz_scatter_8_sigma)
_, prof_scatter_kappa_8_sigma = radial_profile(kappa_scatter_8_sigma)
r_tsz_8_sigma, prof_tsz_8_sigma = radial_profile(tsz_stack_8_sigma)
r_kappa_8_sigma, prof_kappa_8_sigma = radial_profile(kappa_stack_8_sigma)

In [ ]:
print(prof_tsz_8_sigma)

In [ ]:
print((prof_tsz_8_sigma - prof_scatter_tsz_8_sigma))

In [ ]:
print((prof_tsz_8_sigma + prof_scatter_tsz_8_sigma))

In [ ]:
prof_scatter_tsz_8_sigma = prof_scatter_tsz_8_sigma/np.sqrt(23)

In [ ]:
plt.plot(r_tsz_8_sigma * pixel_scale_arcmin, prof_tsz_8_sigma, label = "tSZ")
plt.plot(r_tsz_8_sigma * pixel_scale_arcmin, prof_tsz_8_sigma + prof_scatter_tsz_8_sigma, label = "tSZ scatter", color = 'C3', linestyle = '--')
plt.plot(r_tsz_8_sigma * pixel_scale_arcmin, prof_tsz_8_sigma - prof_scatter_tsz_8_sigma, label = "tSZ - scatter", color = 'C4', linestyle = '--')


In [ ]:
plt.plot(r_tsz_8_sigma * pixel_scale_arcmin, prof_tsz_8_sigma/np.max(prof_tsz_8_sigma), label = "tSZ")
plt.plot(r_tsz_8_sigma * pixel_scale_arcmin, (prof_tsz_8_sigma - prof_scatter_tsz_8_sigma)/(prof_tsz_8_sigma[0]), label = "tSZ - scatter", color = 'C3', linestyle = '--')
plt.plot(r_tsz_8_sigma * pixel_scale_arcmin,  (prof_tsz_8_sigma + prof_scatter_tsz_8_sigma)/np.max(prof_tsz_8_sigma), label = "tSZ - scatter", color = 'C4', linestyle = '--')
#plt.fill_between(r_tsz_8_sigma * pixel_scale_arcmin, -1*(prof_tsz_8_sigma - prof_scatter_tsz_8_sigma)/np.max(np.abs(prof_tsz_8_sigma - prof_scatter_tsz_8_sigma)), (prof_tsz_8_sigma + prof_scatter_tsz_8_sigma)/np.max(prof_tsz_8_sigma+prof_scatter_tsz_8_sigma), alpha=0.3)
plt.plot(r_tsz_0_sigma * pixel_scale_arcmin, prof_tsz_0_sigma/np.max(prof_tsz_0_sigma), label = "tSZ 0-sigma", color = 'C0', linestyle = '--')
plt.plot(r_kappa_8_sigma * pixel_scale_arcmin, prof_kappa_8_sigma/np.max(prof_kappa_8_sigma), label = r"$\kappa$") 
plt.plot(r_kappa_0_sigma * pixel_scale_arcmin, prof_kappa_0_sigma/np.max(prof_kappa_0_sigma), label = r"$\kappa$ 0-sigma", color = 'C1', linestyle = '--')
#plt.yscale('log') 
plt.legend()

In [ ]:
r_kappa_0_sigma, prof_kappa_0_sigma = radial_profile(kappa_stack_0_sigma)
r_kappa_8_sigma, prof_kappa_8_sigma = radial_profile(kappa_stack_8_sigma)
r_tsz_0_sigma, prof_tsz_0_sigma = radial_profile(tsz_stack_0_sigma)
r_tsz_8_sigma, prof_tsz_8_sigma = radial_profile(tsz_stack_8_sigma)

_, scatter_kappa_0_sigma = radial_profile(kappa_scatter_0_sigma)
_, scatter_kappa_8_sigma = radial_profile(kappa_scatter_8_sigma)
_, scatter_tsz_0_sigma = radial_profile(tsz_scatter_0_sigma)
_, scatter_tsz_8_sigma = radial_profile(tsz_scatter_8_sigma)

In [ ]:
kappa_mean_8_sigma = np.mean(kappa_stack_8_sigma, axis = 0)
kappa_0_sigma = np.std(kappa_stack_0_sigma, axis = 0)
kappa_0_sigma_mean = kappa_0_sigma / np.sqrt(len(kappa_stack_0_sigma))
prof_kappa_0_sigma_mean = prof_kappa_0_sigma / np.sqrt(len(kappa_stack_0_sigma))
prof_tsz_0_sigma_mean = prof_tsz_0_sigma / np.sqrt(len(tsz_stack_0_sigma))
tsz_mean_8_sigma = np.mean(tsz_stack_8_sigma, axis = 0)
tsz_0_sigma = np.std(tsz_stack_0_sigma, axis = 0)
tsz_0_sigma_mean = tsz_0_sigma / np.sqrt(len(tsz_stack_0_sigma))


kappa_x = r_kappa_8_sigma * pixel_scale_arcmin
kappa_norm = np.max(prof_kappa_8_sigma)
kappa_y = prof_kappa_8_sigma / kappa_norm
kappa_y_low = (prof_kappa_8_sigma - prof_kappa_0_sigma_mean) / kappa_norm
kappa_y_high = (prof_kappa_8_sigma + prof_kappa_0_sigma_mean) / kappa_norm

plt.figure(figsize=(10, 6))
plt.plot(kappa_x, kappa_y, color='C1', label=r"$\kappa$")
plt.fill_between(kappa_x, kappa_y_low, kappa_y_high, color='C1', alpha=0.3, label=r"$\kappa$ ±1σ Scatter")
plt.legend()
plt.grid(True, alpha=0.3)

In [ ]:
plt.plot(r_kappa_8_sigma * pixel_scale_arcmin, prof_kappa_0_sigma_mean / np.max(prof_kappa_0_sigma_mean), label = r"$\kappa$ 0-sigma mean", color = 'C1', linestyle = '--') 

In [ ]:
halo_props["BoundSubhalo/TotalMass"].convert_to_units(unyt.Solar_Mass)
masses = halo_props["BoundSubhalo/TotalMass"].to('Msun').value

In [ ]:
# --- 2. Isolate the clusters for the 3 bins ---
# The first 3 bins each contain 100 clusters, using the largest clusters.
items_per_bin = 100
num_bins = 3

# Convert mass data to plain numpy floats in solar masses
masses = halo_props["BoundSubhalo/TotalMass"].to('Msun').value
sorted_masses = np.sort(masses)

# Use the largest 300 clusters for the three bins
largest_clusters = sorted_masses[-items_per_bin * num_bins:]

# Split the largest 300 clusters into 3 equal-count bins
quantiles = np.linspace(0, 100, num_bins + 1)
bin_edges = np.percentile(largest_clusters, quantiles)

# Keep the 3-bin structure simple by plotting only these 300 clusters
# with exactly 4 edges (which creates 3 bins)
top_clusters = largest_clusters

# --- 3. Plot the Histogram ---
plt.figure(figsize=(10, 6))
counts, edges, bars = plt.hist(top_clusters, bins=bin_edges, edgecolor='black', color='crimson', alpha=0.8)

# Add exact count labels on top of each bin bar
plt.bar_label(bars, fmt='%d', padding=3, weight='bold')

# Log scale for the x-axis is necessary to visualize wide mass gaps clearly
plt.xscale('log')

plt.xlabel(r'Cluster Mass ($M_{\odot}$)', fontsize=12)
plt.ylabel('Number of Galaxy Clusters', fontsize=12)
plt.grid(axis='both', which='both', linestyle='--', alpha=0.4)

# Set y-limit with padding so labels don't get cut off
plt.ylim(0, max(counts) + 5)

# --- 4. Print out the verification ---
print("Final Bin Breakdown:")
for i in range(len(bin_edges) - 1):
    print(f"Bin {i+1} ({edges[i]:.2e} to {edges[i+1]:.2e}): {int(counts[i])} clusters")

In [ ]:
bin_edges_8sigma = bin_edges

In [ ]:
def get_stack(min_mass, max_mass, map, halo):
    masses = halo["BoundSubhalo/TotalMass"].to('Msun').value
    mask = (masses >= min_mass) & (masses < max_mass)
    selected_halo_x = halo["Lightcone/HaloCentre"][mask][:, 0]
    selected_halo_y = halo["Lightcone/HaloCentre"][mask][:, 1]
    selected_halo_z = halo["Lightcone/HaloCentre"][mask][:, 2] 

    ra_deg_selected = np.arctan2(selected_halo_y, selected_halo_x) * 180 / np.pi
    dec_deg_selected = np.arcsin(selected_halo_z / np.sqrt(selected_halo_x**2 + selected_halo_y**2 + selected_halo_z**2)) * 180 / np.pi

    map_list_selected = []
    for i in range(len(ra_deg_selected)):
        map_patch = hp.gnomview(map, rot=(ra_deg_selected[i], dec_deg_selected[i]), xsize=npix, reso=pixel_scale_arcmin, title="tSZ", return_projected_map=True, cmap='plasma', no_plot=True)
        map_list_selected.append(map_patch)

    map_stack_selected = np.mean(map_list_selected, axis=0)
    map_stack_scatter = np.std(map_list_selected, axis=0)
    return map_stack_selected, map_stack_scatter
    

In [ ]:
#Opening the data for the 8sigma case
basedir="FLAMINGO/L1_m9/fgas-8sigma/healpix_maps/nside_4096"
basename="lightcone0"
shell = lc.ShellArray(basedir, basename, remote_dir=root_dir)

file_path = 'FLAMINGO/L1_m9/fgas-8sigma/integrated_maps/broxterman24/WL_convergence_Euclid_like_nz_Broxterman24_fgas-8sigma_lc0.hdf5'
with hdfstream.open("cosma", file_path) as data_file:
  kappa_map_8sigma = data_file["Convergence"][:]

filename = "FLAMINGO/L1_m9/fgas-8sigma/halo_lightcone/lightcone0/lightcone_halos_0075.hdf5"
halos = lc.HaloLightconeFile(filename, remote_dir=root_dir)

x = h*freq_test*1e9/(T_cmb*k_B)
tSZ_freq_dep_func = x*(np.cosh(x/2)/np.sinh(x/2))-4

with hdfstream.open("cosma", "FLAMINGO/L1_m9/fgas-8sigma/integrated_maps/yang26/lightcone0_shells/lensed_tSZ_rot_same_rot.hdf5") as map_file:
  lensed_tSZ_map_8sigma = map_file["data"][...]
delta_T_tSZ = tSZ_freq_dep_func * lensed_tSZ_map * T_cmb ## in K_CMB
lensed_tSZ_map_8sigma = lensed_tSZ_map_8sigma - np.mean(lensed_tSZ_map_8sigma)  
# List of halo properties to read
properties = ("Lightcone/HaloCentre", "BoundSubhalo/TotalMass", "Lightcone/Redshift")

# Read the data
halo_props_8sigma = halos.read_halos(properties)

In [ ]:
#Sets up the connection to the data and reads in the shell array
root_dir = hdfstream.open("cosma", "/")
basedir="FLAMINGO/L1_m9/L1_m9/healpix_maps/nside_4096"
basename="lightcone0"
shell = lc.ShellArray(basedir, basename, remote_dir=root_dir)

x = h*freq_test*1e9/(T_cmb*k_B)
tSZ_freq_dep_func = x*(np.cosh(x/2)/np.sinh(x/2))-4

with hdfstream.open("cosma", "FLAMINGO/L1_m9/L1_m9/integrated_maps/yang26/lightcone0_shells/lensed_tSZ_rot_same_rot.hdf5") as map_file:
  lensed_tSZ_map_0sigma = map_file["data"][...]
delta_T_tSZ = tSZ_freq_dep_func * lensed_tSZ_map * T_cmb ## in K_CMB
lensed_tSZ_map_0sigma = lensed_tSZ_map_0sigma - np.mean(lensed_tSZ_map_0sigma)

file_path = 'FLAMINGO/L1_m9/L1_m9/integrated_maps/broxterman24/WL_convergence_Euclid_like_nz_Broxterman24_L1_m9_lc0.hdf5'
with hdfstream.open("cosma", file_path) as data_file:
  kappa_map_0sigma = data_file["Convergence"][:]
  
  filename = "FLAMINGO/L1_m9/L1_m9/halo_lightcone/lightcone0/lightcone_halos_0075.hdf5"
halos = lc.HaloLightconeFile(filename, remote_dir=root_dir)
# List of halo properties to read
properties = ("Lightcone/HaloCentre", "BoundSubhalo/TotalMass", "Lightcone/Redshift")

# Read the data
halo_props_0sigma = halos.read_halos(properties)

In [ ]:
for i in range(len(bin_edges_0sigma)-1):
    min_mass_0_sigma = bin_edges_0sigma[i]
    max_mass_0_sigma = bin_edges_0sigma[i+1]
    map_stack_0sigma, map_stack_scatter_0sigma = get_stack(min_mass_0_sigma, max_mass_0_sigma, lensed_tSZ_map_0sigma, halo_props_0sigma)
    kappa_stack_0sigma, kappa_stack_scatter_0sigma = get_stack(min_mass_0_sigma, max_mass_0_sigma, kappa_map_0sigma, halo_props_0sigma)
    r_kappa_0_sigma, prof_kappa_0sigma = radial_profile(kappa_stack_0sigma)
    _, scatter_kappa_0sigma = radial_profile(kappa_stack_scatter_0sigma)
    r_tsz_0_sigma, map_stack_0sigma = radial_profile(map_stack_0sigma)
    _, scatter_tsz_0_sigma = radial_profile(map_stack_scatter_0sigma)

    min_mass_8_sigma = bin_edges_8sigma[i]
    max_mass_8_sigma = bin_edges_8sigma[i+1]
    map_stack_8sigma, map_stack_scatter_8sigma = get_stack(min_mass_8_sigma, max_mass_8_sigma, lensed_tSZ_map_8sigma, halo_props_8sigma)
    kappa_stack_8sigma, kappa_stack_scatter_8sigma = get_stack(min_mass_8_sigma, max_mass_8_sigma, kappa_map_8sigma, halo_props_8sigma)
    r_kappa_8_sigma, prof_kappa_8_sigma = radial_profile(kappa_stack_8_sigma)
    _, scatter_kappa_8sigma = radial_profile(kappa_stack_scatter_8sigma)
    r_tsz_8_sigma, map_stack_8sigma = radial_profile(map_stack_8sigma)
    _, scatter_tsz_8_sigma = radial_profile(map_stack_scatter_8sigma)
    
    scatter_tsz_8_sigma = scatter_tsz_8_sigma/np.sqrt(len(map_stack_8sigma))
    scatter_tsz_0_sigma = scatter_tsz_0_sigma/np.sqrt(len(map_stack_0sigma))
    plt.figure(figsize=(10, 6))
    plt.plot(r_tsz_0_sigma * pixel_scale_arcmin, map_stack_0sigma/np.max(map_stack_0sigma), label = "tSZ 0-sigma bin " + str(i+1), linestyle = '--', color = 'C0' )
    plt.plot(r_kappa_0_sigma * pixel_scale_arcmin, prof_kappa_0sigma/np.max(prof_kappa_0sigma), label = "Kappa 0-sigma bin " + str(i+1), linestyle = '--', color = 'C1' )
    plt.plot(r_tsz_8_sigma * pixel_scale_arcmin, map_stack_8sigma/np.max(map_stack_8sigma), label = "tSZ 8-sigma bin " + str(i+1), color = 'C0' )
    plt.plot(r_kappa_8_sigma * pixel_scale_arcmin, prof_kappa_8_sigma/np.max(prof_kappa_8_sigma), label = "Kappa 8-sigma bin " + str(i+1), color = 'C1' )
    plt.fill_between(r_tsz_0_sigma * pixel_scale_arcmin, (map_stack_0sigma - scatter_tsz_0_sigma)/np.max(map_stack_0sigma), (map_stack_0sigma + scatter_tsz_0_sigma)/np.max(map_stack_0sigma), alpha=0.3)
    plt.fill_between(r_tsz_8_sigma * pixel_scale_arcmin, (map_stack_8sigma - scatter_tsz_8_sigma)/np.max(map_stack_8sigma), (map_stack_8sigma + scatter_tsz_8_sigma)/np.max(map_stack_8sigma), alpha=0.3)
    plt.xscale('log')
    plt.legend()

In [ ]:
71, 67, 64